# Autonomous electric vehicle — ThunderGraph model demo

This notebook sketches a **Level-4-capable autonomous electric vehicle (AEV)** using **thundergraph-model**: **system-level requirements** (with optional acceptance `expr`), **derived attributes** and **part constraints** (unitflow expressions), top-level subsystems, structural connections, and **Phase 6** behavior (state machines, guards, sequences, fork/join, item flow, scenarios).

**Scope:** Mission and architecture down to **named subsystems** — not ECU-level wiring. The goal is to show: `parameter` / `attribute(expr=…)` / `constraint(expr=…)` on parts, **`requirement` + `allocate` only on the vehicle system**, `compile_graph` + `Evaluator`, `summarize_requirement_satisfaction`, plus `port` + `connect` + `item_kind`, discrete behavior, `RunContext`, and trace validation.

**Run:** From the `thundergraph-model` directory: `uv sync --all-groups` (uses **`thundergraph-model/.venv`**, separate from the monorepo root venv). Run cells **top to bottom**, or execute headless:
`uv run jupyter nbconvert --to notebook --execute notebooks/aev_thundergraph_demo.ipynb --output aev_thundergraph_demo_executed.ipynb --output-dir notebooks`.
Optional UI: `uv run jupyter lab notebooks/aev_thundergraph_demo.ipynb`.


## 1. High-level requirements

| ID | Statement |
|----|-----------|
| **R-SAFE** | The vehicle shall maintain a safe state when supervisory checks fail. |
| **R-FLOW** | Scene observations shall reach motion planning across structural connections. |
| **R-TRACTION** | Traction power shall be modulated for driver intent and pack health. |
| **R-AUTO** | Autonomy engagement shall require explicit readiness (supervisor / command gating). |

On **`AutonomousVehicle`**, each row becomes a **`model.requirement(...)`** and is **allocated** to a subsystem with **`model.allocate`**. Acceptance **`expr=`** (where used) is evaluated on the **allocated part** after **`compile_graph` + `evaluate`** (see §3). **Do not** declare requirements inside part `define` blocks — keep them at the system/configuration level.


## 2. Architecture (top-level subsystems)

See **`aev_architecture.mmd`** (raw Mermaid for CLI/tools).

- **Powertrain** — **derived** `shaft_power = cmd_torque × shaft_speed`, **part constraint** (shaft power ≥ 0), discrete **states** (`standby` → `propulsion`), effects that bind **SOC**.
- **Motion** — **guarded** transition into `autonomous` when `auton_cmd_ok` meets the supervisor threshold; system-level **requirement** `expr` can mirror that policy.
- **Perception → Planner** — **structural** `connect` with **`item_kind` `Observation`**; runtime uses **`emit_item`** to deliver a payload and dispatch the receiver’s matching **event**.
- **Safety** — **`sequence`** (ordered checks) and **`fork_join`** (v0: deterministic **serial** branches, then continuation), per library semantics.


In [7]:
# nbconvert often sets cwd to `notebooks/`; editable install also works after `uv sync`.
from __future__ import annotations

import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_pkg_root = next(
    (p for p in (_cwd, _cwd.parent) if (p / "tg_model" / "__init__.py").is_file()),
    _cwd,
)
if str(_pkg_root) not in sys.path:
    sys.path.insert(0, str(_pkg_root))

from unitflow import Quantity
from unitflow.catalogs.si import N, m, rad, s
from unitflow.core.units import Unit

# String metadata like unit="1" breaks unitflow symbols in relational expr; use dimensionless Unit.
DIMLESS = Unit.dimensionless()

from tg_model.model.elements import Part, System
from tg_model.execution.configured_model import instantiate
from tg_model.execution.evaluator import Evaluator
from tg_model.execution.graph_compiler import compile_graph
from tg_model.execution.requirements import summarize_requirement_satisfaction
from tg_model.execution.run_context import RunContext
from tg_model.execution.validation import validate_graph
from tg_model.execution.instances import PartInstance
from tg_model.execution.behavior import (
    BehaviorTrace,
    DecisionDispatchOutcome,
    DispatchOutcome,
    behavior_authoring_projection,
    behavior_trace_to_records,
    dispatch_decision,
    dispatch_event,
    dispatch_fork_join,
    dispatch_sequence,
    emit_item,
    validate_scenario_trace,
)


In [8]:
def _reset_aev_types() -> None:
    for cls in (
        PowertrainPart,
        MotionPart,
        PerceptionStackPart,
        PlannerPart,
        SafetySupervisorPart,
        AutonomousVehicle,
    ):
        cls._reset_compilation()


class PowertrainPart(Part):
    """Battery + inverter + traction: derived shaft power + part constraints; discrete traction FSM."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        # Requirements are NOT declared on parts — only parameters, attributes, constraints, behavior.
        cmd_torque = model.parameter("cmd_torque", unit=N * m)
        shaft_speed = model.parameter("shaft_speed", unit=rad / s)
        shaft_power = model.attribute(
            "shaft_power",
            unit=N * m / s,
            expr=cmd_torque * shaft_speed,
        )
        model.constraint(
            "shaft_power_non_negative",
            expr=shaft_power >= Quantity(0, N * m / s),
        )
        soc = model.parameter("soc", unit="1")
        st = model.state("standby", initial=True)
        pr = model.state("propulsion")
        flt = model.state("fault")
        ev_enable = model.event("enable_traction")
        ev_fault = model.event("fault_event")
        ev_reset = model.event("reset")

        def apply_torque(c: RunContext, p: PartInstance) -> None:
            c.bind_input(p.soc.stable_id, 0.92)

        model.action("apply_torque", effect=apply_torque)
        model.transition(st, pr, on=ev_enable, effect="apply_torque")
        model.transition(pr, flt, on=ev_fault)
        model.transition(st, flt, on=ev_fault)
        model.transition(flt, st, on=ev_reset)


class MotionPart(Part):
    """Operating mode: manual vs autonomous with a guard on engagement."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        auton_cmd_ok = model.parameter("auton_cmd_ok", unit=DIMLESS)
        min_supervisor_score = model.parameter("min_supervisor_score", unit=DIMLESS)
        manual = model.state("manual", initial=True)
        auto = model.state("autonomous")
        engage = model.event("engage_auto")
        disengage = model.event("disengage")
        g = model.guard(
            "auton_ok",
            predicate=lambda c, p: c.get_value(p.auton_cmd_ok.stable_id) >= 1.0,
        )
        model.transition(manual, auto, on=engage, guard=g)
        model.transition(auto, manual, on=disengage)


class PerceptionStackPart(Part):
    """Publishes fused scene observations on a structural port."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.port("scene_out", direction="out")
        model.item_kind("Observation")


class PlannerPart(Part):
    """Consumes observations at a port; transition on item-carrying event."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.port("fusion_in", direction="in")
        fused_stamp = model.parameter("fused_stamp", unit=DIMLESS)
        min_valid_stamp = model.parameter("min_valid_stamp", unit=DIMLESS)
        model.constraint(
            "fusion_stamp_ge_threshold",
            expr=fused_stamp >= min_valid_stamp,
        )

        def ingest(c: RunContext, p: PartInstance) -> None:
            pl = c.peek_item_payload(p.path_string, "Observation")
            if pl is not None:
                c.bind_input(p.fused_stamp.stable_id, float(pl))

        model.action("ingest", effect=ingest)
        obs = model.event("Observation")
        idle = model.state("idle", initial=True)
        busy = model.state("busy")
        model.transition(idle, busy, on=obs, effect="ingest")


class SafetySupervisorPart(Part):
    """Redundant checks: linear preflight then fork/join style dual path (v0 serial)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.action("check_comms", effect=lambda c, p: None)
        model.action("check_estop", effect=lambda c, p: None)
        model.action("check_sensor_a", effect=lambda c, p: None)
        model.action("check_sensor_b", effect=lambda c, p: None)
        model.action("mark_ready", effect=lambda c, p: None)
        model.sequence("preflight", steps=["check_comms", "check_estop"])
        model.fork_join(
            "dual_check",
            branches=[["check_sensor_a"], ["check_sensor_b"]],
            then_action="mark_ready",
        )


class AutonomousVehicle(System):
    """Root system: all requirements here; allocate to subsystems; connect + scenario."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        pt = model.part("powertrain", PowertrainPart)
        motion = model.part("motion", MotionPart)
        perc = model.part("perception", PerceptionStackPart)
        plan = model.part("planner", PlannerPart)
        safe = model.part("safety", SafetySupervisorPart)

        r_safe = model.requirement(
            "req_safe",
            "Vehicle shall fail safe when supervisory checks fail.",
        )
        r_flow = model.requirement(
            "req_perception_flow",
            "Fusion stamp shall meet minimum threshold (perception → planner path).",
            expr=plan.fused_stamp >= plan.min_valid_stamp,
        )
        r_traction = model.requirement(
            "req_traction",
            "Traction mechanical power shall be non-negative under command.",
            expr=pt.shaft_power >= Quantity(0, N * m / s),
        )
        r_auto = model.requirement(
            "req_autonomy",
            "Autonomy engagement shall require supervisor readiness (cmd ≥ threshold).",
            expr=motion.auton_cmd_ok >= motion.min_supervisor_score,
        )
        model.allocate(r_safe, safe)
        model.allocate(r_flow, plan)
        model.allocate(r_traction, pt)
        model.allocate(r_auto, motion)
        model.connect(
            source=perc.scene_out,
            target=plan.fusion_in,
            carrying="Observation",
        )
        model.scenario(
            "sensor_to_plan",
            expected_event_order=[],
            expected_interaction_order=[
                ("powertrain", "enable_traction"),
                ("motion", "engage_auto"),
                ("planner", "Observation"),
            ],
            expected_item_kind_order=["Observation"],
        )


_reset_aev_types()


## 3. Derived attributes, part constraints, and requirement acceptance

**Subsystems** below declare only **parameters**, **attributes** (with **`expr=`** where derived), **constraints**, ports, and behavior — **not** formal requirements.

**Requirements** live on **`AutonomousVehicle`**: **`model.allocate`** for traceability, and optional **`expr=`** for executable acceptance (Phase 7). The same **`compile_graph` → `Evaluator.evaluate`** run evaluates **part-owned constraints** and **requirement acceptance** checks; **`summarize_requirement_satisfaction`** reports pass/fail per allocated requirement ( **`all_passed`** is **False** if there are zero acceptance checks in the graph).

`req_safe` is **text + allocate only** (no `expr`), so it does not create an extra acceptance node.

In [12]:
_reset_aev_types()
cm = instantiate(AutonomousVehicle)
graph, handlers = compile_graph(cm)
v = validate_graph(graph)
assert v.passed, v.failures

evalr = Evaluator(graph, compute_handlers=handlers)
ctxv = RunContext()
result = evalr.evaluate(
    ctxv,
    inputs={
        cm.powertrain.cmd_torque.stable_id: Quantity(800, N * m),
        cm.powertrain.shaft_speed.stable_id: Quantity(120, rad / s),
        cm.powertrain.soc.stable_id: 0.88,
        cm.motion.auton_cmd_ok.stable_id: 1.0,
        cm.motion.min_supervisor_score.stable_id: 1.0,
        cm.planner.fused_stamp.stable_id: 20_250_324.0,
        cm.planner.min_valid_stamp.stable_id: 1.0,
    },
)

sp = ctxv.get_value(cm.powertrain.shaft_power.stable_id)
print("shaft_power (cmd_torque × shaft_speed):", sp)
print("Constraint results (part + requirement acceptance):", len(result.constraint_results))
for cr in result.constraint_results:
    print(" ", cr)

summary = summarize_requirement_satisfaction(result)
print(
    "Requirement acceptance summary:",
    summary.check_count,
    "checks | all_passed:",
    summary.all_passed,
)
for row in summary.results:
    tag = "PASS" if row.passed else "FAIL"
    print(" ", row.requirement_path, "→", row.allocation_target_path, "|", tag)

shaft_power (cmd_torque × shaft_speed): 96000 m^2 * kg / s^3
Constraint results (part + requirement acceptance): 5
  <ConstraintResult: reqcheck:AutonomousVehicle.req_autonomy@a79341b7-f042-5574-b290-2825366eea0d PASS requirement='AutonomousVehicle.req_autonomy' target='AutonomousVehicle.motion'>
  <ConstraintResult: AutonomousVehicle.planner.fusion_stamp_ge_threshold PASS>
  <ConstraintResult: reqcheck:AutonomousVehicle.req_perception_flow@162773f3-7b48-56e9-a845-5203a0a74f36 PASS requirement='AutonomousVehicle.req_perception_flow' target='AutonomousVehicle.planner'>
  <ConstraintResult: AutonomousVehicle.powertrain.shaft_power_non_negative PASS>
  <ConstraintResult: reqcheck:AutonomousVehicle.req_traction@a3636525-791f-5400-baea-243e013b497f PASS requirement='AutonomousVehicle.req_traction' target='AutonomousVehicle.powertrain'>
Requirement acceptance summary: 3 checks | all_passed: True
  AutonomousVehicle.req_autonomy → AutonomousVehicle.motion | PASS
  AutonomousVehicle.req_percep

## 4. Compile and inspect structure

The compiled artifact includes **nodes**, **edges** (**connect** + multiple **allocate**), and **behavior** transition summaries.


In [13]:
_reset_aev_types()
av = AutonomousVehicle.compile()
print("Owner:", av["owner"])
print("Top-level parts:", [k for k, v in av["nodes"].items() if v["kind"] == "part"])
print("Edges:", len(av["edges"]))
for e in av["edges"]:
    print(" ", e["kind"], e.get("carrying") or "")
print("Behavior transitions (sample):", av["behavior_transitions"][:3], "...")


Owner: __main__.AutonomousVehicle
Top-level parts: ['powertrain', 'motion', 'perception', 'planner', 'safety']
Edges: 5
  allocate 
  allocate 
  allocate 
  allocate 
  connect Observation
Behavior transitions (sample): [] ...


## 5. Authoring projection (diagram / export hook)

`behavior_authoring_projection` returns a JSON-friendly summary of discrete behavior nodes per **part type**.


In [14]:
_reset_aev_types()
proj = behavior_authoring_projection(SafetySupervisorPart)
print({k: proj[k] for k in ("sequences", "fork_joins", "actions")})


{'sequences': ['preflight'], 'fork_joins': ['dual_check'], 'actions': ['check_comms', 'check_estop', 'check_sensor_a', 'check_sensor_b', 'mark_ready']}


## 6. Instantiate and run a scripted mission

Same **`RunContext`** holds parameters and discrete state for all parts. Order below: traction → engage autopilot (guard) → safety **sequence** + **fork_join** → **emit_item** to planner. Bind **`min_supervisor_score`** so the motion guard matches the acceptance policy used in §3.


In [15]:
_reset_aev_types()
cm = instantiate(AutonomousVehicle)
ctx = RunContext()
ctx.bind_input(cm.motion.auton_cmd_ok.stable_id, 1.0)
ctx.bind_input(cm.motion.min_supervisor_score.stable_id, 1.0)
ctx.bind_input(cm.powertrain.soc.stable_id, 0.9)

tr = BehaviorTrace()

r1 = dispatch_event(ctx, cm.powertrain, "enable_traction", trace=tr)
r2 = dispatch_event(ctx, cm.motion, "engage_auto", trace=tr)
dispatch_sequence(ctx, cm.safety, "preflight", trace=tr)
dispatch_fork_join(ctx, cm.safety, "dual_check", trace=tr)
out = emit_item(ctx, cm, cm.perception.scene_out, "Observation", 20250324.0, trace=tr)

print("Traction dispatch:", r1.outcome)
print("Autopilot dispatch:", r2.outcome)
print("emit_item branch results:", [x.outcome for x in out])
print("Planner fused_stamp:", ctx.get_value(cm.planner.fused_stamp.stable_id))
print("Powertrain soc:", ctx.get_value(cm.powertrain.soc.stable_id))
print("Motion mode:", ctx.get_active_behavior_state(cm.motion.path_string))


Traction dispatch: fired
Autopilot dispatch: fired
emit_item branch results: [<DispatchOutcome.FIRED: 'fired'>]
Planner fused_stamp: 20250324.0
Powertrain soc: 0.92
Motion mode: autonomous


## 7. Optional: decision + merge (single-part pattern)

Not used on the vehicle root above; **`MotionPart`** could use a **`decision`** node instead of a guarded transition for the same intent. Here is a minimal **temperature routing** example on a throwaway part (shows `DecisionDispatchResult`).


In [16]:
class ThermalRouter(Part):
    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.parameter("temp", unit="1")
        hot = model.guard("hot", predicate=lambda c, p: c.get_value(p.temp.stable_id) > 40.0)
        model.action("cool", effect=lambda c, p: c.bind_input(p.temp.stable_id, 25.0))
        model.action("hold", effect=lambda c, p: None)
        model.decision("route", branches=[(hot, "cool")], default_action="hold")


ThermalRouter._reset_compilation()
ctx2 = RunContext()
cm_tr = instantiate(ThermalRouter)
ctx2.bind_input(cm_tr.temp.stable_id, 55.0)
tr2 = BehaviorTrace()
r = dispatch_decision(ctx2, cm_tr, "route", trace=tr2)
print(r.outcome, r.chosen_action, bool(r))
assert r.outcome == DecisionDispatchOutcome.ACTION_RAN


action_ran cool True


## 8. Scenario validation (partial contract)

`validate_scenario_trace` bundles checks: **global transition order** for `expected_interaction_order` (state-machine steps only), **item kind order**, etc. Re-run after changing the trace.


In [17]:
_reset_aev_types()

cm = instantiate(AutonomousVehicle)
ctx = RunContext()
ctx.bind_input(cm.motion.auton_cmd_ok.stable_id, 1.0)
ctx.bind_input(cm.motion.min_supervisor_score.stable_id, 1.0)
ctx.bind_input(cm.powertrain.soc.stable_id, 0.9)
tr = BehaviorTrace()
dispatch_event(ctx, cm.powertrain, "enable_traction", trace=tr)
dispatch_event(ctx, cm.motion, "engage_auto", trace=tr)
dispatch_sequence(ctx, cm.safety, "preflight", trace=tr)
dispatch_fork_join(ctx, cm.safety, "dual_check", trace=tr)
emit_item(ctx, cm, cm.perception.scene_out, "Observation", 20250324.0, trace=tr)

ok, errors = validate_scenario_trace(
    definition_type=AutonomousVehicle,
    scenario_name="sensor_to_plan",
    part_path=cm.path_string,
    trace=tr,
    root=cm.root,
)
print("Scenario OK:", ok)
print("Errors:", errors)


Scenario OK: True
Errors: []


## 9. Flat trace export

`behavior_trace_to_records` merges transition, item, sequence, fork/join, decision, and merge steps for tooling. This cell rebuilds the same trace as section 8 so it runs **standalone** (you do not need prior cells in session memory).


In [18]:
_reset_aev_types()
cm = instantiate(AutonomousVehicle)
ctx = RunContext()
ctx.bind_input(cm.motion.auton_cmd_ok.stable_id, 1.0)
ctx.bind_input(cm.motion.min_supervisor_score.stable_id, 1.0)
ctx.bind_input(cm.powertrain.soc.stable_id, 0.9)
tr = BehaviorTrace()
dispatch_event(ctx, cm.powertrain, "enable_traction", trace=tr)
dispatch_event(ctx, cm.motion, "engage_auto", trace=tr)
dispatch_sequence(ctx, cm.safety, "preflight", trace=tr)
dispatch_fork_join(ctx, cm.safety, "dual_check", trace=tr)
emit_item(ctx, cm, cm.perception.scene_out, "Observation", 20250324.0, trace=tr)

records = behavior_trace_to_records(tr)
for rec in records[:8]:
    print(rec)
print("...", len(records), "records total")


{'kind': 'transition', 'step_index': 0, 'part_path': 'AutonomousVehicle.powertrain', 'event_name': 'enable_traction', 'from_state': 'standby', 'to_state': 'propulsion', 'effect_name': 'apply_torque'}
{'kind': 'transition', 'step_index': 1, 'part_path': 'AutonomousVehicle.motion', 'event_name': 'engage_auto', 'from_state': 'manual', 'to_state': 'autonomous', 'effect_name': None}
{'kind': 'sequence', 'step_index': 2, 'part_path': 'AutonomousVehicle.safety', 'sequence_name': 'preflight'}
{'kind': 'fork_join', 'step_index': 3, 'part_path': 'AutonomousVehicle.safety', 'block_name': 'dual_check'}
{'kind': 'item_flow', 'step_index': 4, 'source_port_path': 'AutonomousVehicle.perception.scene_out', 'target_port_path': 'AutonomousVehicle.planner.fusion_in', 'item_kind': 'Observation', 'payload': 20250324.0}
{'kind': 'transition', 'step_index': 5, 'part_path': 'AutonomousVehicle.planner', 'event_name': 'Observation', 'from_state': 'idle', 'to_state': 'busy', 'effect_name': 'ingest'}
... 6 records